# Group 5: A crowd monitoring dashboard for SAIL Amsterdam 2025

*TIL6022 Python Programming, project proposal, 2 October 2026*

Members:

- Andrea Della Corte (GitHub: `andreadellacorte3`)
- Mateo Gonzalez (GitHub: `mateogonzalezbrown`)
- Daniele Bini (GitHub: `Danielebini27`)
- Roberto Cortes Camilo De Sales (GitHub: `cortesjose36-png`)
- Davide Pighin (GitHub: `Dpighin`)

# Introduction

SAIL Amsterdam 2025 brought tall ships and about 2.5 million visitors to the IJ between 20 and 24 August. With so many people on the same streets, crowd managers need to see quickly where it is getting busy and whether they have to act. A real-time monitoring dashboard puts that information on one screen.

During the event the City of Amsterdam counted pedestrians with sensors around the IJ. Raw counts are not enough on their own: a sensor can fail, the same number of people means more in a narrow street than on a wide quay, and crowds also react to things outside the event, such as rain or the arrival of a train. In this project we build a dashboard from these counts and study some of these other factors. The project is joint with TIL4030 Research and Design Methods, where the same group makes the conceptual design of the dashboard.

# Research Objectives

The objective is to turn the pedestrian counts collected during SAIL 2025 into a dashboard that crowd managers can trust and read quickly. The main research question is:

*How can the pedestrian counts measured around the IJ during SAIL 2025 be turned into a real-time dashboard that shows crowd managers where and when walkways came close to critical conditions, and how do other factors affect crowd behaviour?*

1. How reliable are the sensor data, and how can we tell a sensor that has stopped working from a street that is really empty?
2. What would a dashboard that replays 20 to 24 August 2025 on a map have shown crowd managers at the busiest moments of the event?
3. How much busier were the streets during SAIL 2025 than in a normal week in August, and how can a second dashboard tab show the difference?
4. Can the dashboard show how incoming precipitation has affected pedestrian flows?
5. Do train arrivals and departures lead to spikes in crowd counts, and are they a determining factor?

## Geographical and temporal scale

The study covers the area around the IJ in Amsterdam where the sensors in our dataset are placed, about 4 km by 3 km: the walkways and ferry landings behind Amsterdam Centraal, the banks of the IJ and part of the old city centre. In time, it covers the five days of SAIL 2025, 20 to 24 August 2025, and, for sub-question 3, a normal week in August 2023.

# Contribution Statement

All five members chose the research questions and wrote this proposal together. In the next phase each member leads one sub-question, and the others review the work through pull requests on GitHub.

# Available data

The provided dataset includes the following files:

- `sensor-location.xlsx`: locations of the pedestrian crowd-monitoring sensors referenced in the flow dataset below.
- `SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv`: pedestrian crowd flow counts from the LVMA (Langzaam Verkeer Monitoringssysteem Amsterdam) sensor network, aggregated in 3-minute intervals, 20-24 August 2025.
- `NWB_roads.zip`: NWB (Nationaal Wegen Bestand) road network shapefile data, used to identify and spatially reference road segments (road IDs).
- `TomTom_data_20-24Aug2025.csv`: TomTom car traffic flow data, 20-24 August 2025. Can be joined to the NWB road network via road ID.
- `Vesselposition_data_20-24Aug2025.csv`: vessel position data, 20-24 August 2025.

## Related open data

- LVMA crowd monitoring map (SAIL pedestrian sensor data): https://maps.amsterdam.nl/lvma/
- Amsterdam open geodata portal (search "Langzaam Verkeer Monitoringssysteem" for the LVMA dataset): https://maps.amsterdam.nl/open_geodata/
- Crowd monitor archive of the City of Amsterdam, with hourly counts per sensor from 2019 to March 2024. We use it for the normal week in sub-question 3: https://api.data.amsterdam.nl/v1/crowdmonitor/passanten/

## Additional data

- Weather data: the precipitation data is crucial in studying how precipitation can affect the crowd's behaviour. We use the hourly data of the Royal Netherlands Meteorological Institute for its station at Schiphol: https://www.knmi.nl/nederland-nu/klimatologie/uurgegevens
- NS departures and arrivals from Amsterdam Centraal: these data can be linked to pedestrian flow and be used to determine whether an increase in flow is due to a train arrival or departure. We take them from the open train archive of Rijden de Treinen: https://www.rijdendetreinen.nl/en/open-data/train-archive

# Data Pipeline

1. Load and clean (sub-question 1). Read the count file and turn it from one column per sensor and direction into a long table (time, sensor, direction, count).
2. Flag missing data (sub-question 1). Mark a sensor as offline (broken) when both directions report zero for a long period (we start with one hour) while sensors of the same type are counting people. The dashboard shows offline sensors.
3. Compute indicators. Flow per metre for each direction and for both directions together. Rank the locations by count and by flow per metre and compare the two rankings.
4. Compare types of locations. The course slides show that the city centre, the SAIL walkways and the ferry landings behave very differently. We compare their daily profiles and peak times, and test whether one threshold fits all of them, starting from the walkway level-of-service scale of Fruin (1971). Ferry landings may need a longer averaging window because people arrive in waves.
5. Build the dashboard (sub-question 2). A Streamlit app with a time slider that replays 20 to 24 August, so that the user can follow how the situation evolves: a map with each sensor coloured by its level and marked when offline, a time series of the selected location with its threshold, and a list of active warnings. Every screen states which moment the data belong to.
6. Test an early warning. Compare a simple trend indicator with a baseline that assumes nothing changes. For each method we count how many threshold crossings it warns about in advance and how many of its warnings are false.
7. Compare with a normal week (sub-question 3). Download the hourly counts of a normal week in August 2023 from the crowd monitor archive, for the sensors that appear in both datasets, and show the difference in a second dashboard tab.
8. Add the weather (sub-question 4). Join the hourly precipitation to the counts and look at how the flows change during and after rain.
9. Add the trains (sub-question 5). Count the train arrivals and departures at Amsterdam Centraal over time and compare them with the counts of the sensors near the station.